# Leer cuantiles, rechazos y efectos de varianza

**Pregunta:** ¿Cómo afectan los denominadores de cuantiles y las estimaciones finitas de Sobol a la interpretación?

Nivel: 2. Requisitos previos: lecciones anteriores del tema. Lectura: 30 minutos.



## Objetivos

Separar cuantiles de salidas resueltas y factibilidad sobre intentos, y examinar efectos Sobol con dos presupuestos. Cada resumen se calcula mediante el motor a la semilla declarada.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Importar las herramientas fijadas y resolver las entradas relativas a la carpeta del notebook.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
factor_labels = {
    "work_index": "Índice de trabajo",
    "head_grade": "Ley de cabeza",
    "liberation_size": "Tamaño de liberación",
    "floatability": "Flotabilidad",
}
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. El catálogo define mineral y circuito nominales del pórfido blando. engine/data/constants.json declara semiamplitudes uniformes independientes y tolerancias de sensibilidad. Se entregan cantidades de muestras explícitas y semilla 20261031 a los métodos de incertidumbre. No se carga contrato operativo, filtro desplegado, tabla archivada ni distribución medida. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

Primero se calcula distribución de 32 sorteos con denominadores de intentos y resueltos. Después se ejecuta Saltelli/Sobol de producción con N=8 y N=16, iguales semilla y rangos. Cuatro factores requieren 48 y 96 evaluaciones. Las filas híbridas no pueden eliminarse tras un rechazo: se informa diseño inválido. Primer orden y total responden preguntas de varianza distintas y bootstrap cuantifica estabilidad finita. Una salida constante se reconoce numéricamente, sin ordenar redondeo.

Fijar el punto nominal para examinar dos resúmenes distintos. Propagación estima distribución condicional; Sobol descompone varianza entre factores independientes. Ambos usan rangos minerales de autoría y motor exacto, pero matrices y preguntas diferentes. Partir de recuperación nominal para dar una referencia física común a distribución y contribuciones.

In [ ]:
# Partir del circuito de pórfido blando y su mineral nominal.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.uncertainty import uncertainty, sensitivity, perturbed

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
print("Recuperación nominal (%):", round(nominal.metrics["recovery_pct"], 3))


Calcular la respuesta inferior de escala de liberación para mostrar cómo entra el factor al modelo. Cambiar accesibilidad, no la salida directamente. El control ayuda si luego un efecto total resulta grande: distingue una vía física plausible de una variable con rango ancho. No estima por sí solo participación de varianza global.

In [ ]:
# Resolver el factor inferior de tamaño de liberación de 0.9, manteniendo los demás factores en
# uno.
low_ore, low_point = perturbed(case, case.nominal, {"liberation_size": 0.9})
low = simulate(low_ore, case.plant, low_point)
print("Factor de liberación:", 0.9)
print("Recuperación condicional (%):", round(low.metrics["recovery_pct"], 3))


Calcular independientemente la respuesta superior. Mantener los demás factores en uno hace interpretables los controles localmente. Su diferencia del nominal no es confianza ni derivada promediada sobre todo el dominio. Interacciones con ley, dureza y flotabilidad aparecen al variar también esos factores. La comparación motiva un diseño global, sin reemplazarlo.

In [ ]:
# Resolver el factor superior de tamaño de liberación de 1.1, manteniendo los demás factores en
# uno.
high_ore, high_point = perturbed(case, case.nominal, {"liberation_size": 1.1})
high = simulate(high_ore, case.plant, high_point)
print("Factor de liberación:", 1.1)
print("Recuperación condicional (%):", round(high.metrics["recovery_pct"], 3))


Propagar el hipercubo de 32 sorteos y conservar salidas y restricciones. P05 y P95 son cuantiles empíricos de estados resueltos. La probabilidad cuenta rechazos como fallas y puede empeorar aunque el intervalo condicional parezca inofensivo. Informar denominador real y motivos de rechazo cuando otro lector compare intervalos entre puntos operativos.

In [ ]:
# Propagar 32 sorteos de hipercubo latino por el motor de producción con semilla fija.
experiment = uncertainty(case, case.nominal, samples=32, seed=20261031)
print("Sorteos resueltos:", len(experiment["solved"]))
print("Sorteos rechazados:", experiment["refused"])
print("Semiamplitudes de factores:", experiment["inputs"])
summary = pd.DataFrame(experiment["outputs"]).T[["p05", "p50", "p95", "std"]]
display(summary.round(4))
print(
    "Probabilidad de todas las restricciones:",
    round(experiment["probabilities"]["all_constraints"], 4),
)


Mostrar la distribución antes de reducirla a momentos. Desviación estándar e intervalo P05-P95 conservan aspectos distintos de forma. El histograma puede revelar asimetría por no linealidad o límites, aunque 32 sorteos no modelan colas detalladamente. Su marca nominal es referencia del motor y no recuperación esperada observada.

In [ ]:
# Graficar sorteos de recuperación y referencia nominal para mostrar la distribución sin reducirla
# a tres cuantiles..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
recovery = np.asarray(experiment["outputs"]["recovery_pct"]["values"])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.hist(recovery, bins=8, edgecolor="white")
ax.axvline(nominal.metrics["recovery_pct"], color="darkred", label="Nominal")
ax.set(
    xlabel="Recuperación (%)",
    ylabel="Sorteos\nresueltos\n(cantidad)",
    title="Propagación de incertidumbre mineral",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)


Leer la forma condicionada a uniformes independientes. Una distribución de planta necesitaría dependencia geológica y calibración que la estratificación no entrega. Concentrarse en un extremo puede provenir de restricciones, no de un régimen mineral. Conservar definiciones: ensanchar un rango cambia importancia aparente de otros mediante la varianza total.

Ejecutar Saltelli con ocho filas base. Filas híbridas separan contribución individual y total con interacciones. Todas deben resolverse físicamente; eliminar un rechazo rompe la matriz del estimador. Por ello producción rehúsa informar índices si un sorteo falla. Conservar ese resultado científico sin rellenar el diseño con salidas nominales.

In [ ]:
# Calcular la descomposición Saltelli/Sobol con base N=8.
small_indices = sensitivity(case, case.nominal, base_samples=8, seed=20261031)
print("Filas base Sobol:", small_indices["base_samples"])
print("Evaluaciones del motor Sobol:", small_indices["evaluations"])
print("Estado Sobol:", small_indices["status"])
if small_indices["status"] != "computed":
    raise RuntimeError("El diseño de sensibilidad contiene sorteos rechazados")
effects = small_indices["indices"]["recovery_pct"]
if effects.get("constant"):
    raise RuntimeError("La recuperación es constante con la tolerancia declarada")
display(pd.DataFrame({key: effects[key] for key in ("S1", "ST", "ST_conf")}))


Duplicar a dieciséis filas con iguales semilla y rangos. Cambiar presupuesto, no modelo físico. Estimaciones finitas pueden moverse mucho con tamaños pequeños; un primer orden negativo pequeño es error de muestreo, no varianza física negativa. Las semiamplitudes bootstrap cuantifican variabilidad del estimador, no incertidumbre de planta real.

In [ ]:
# Calcular la descomposición Saltelli/Sobol con base N=16.
larger_indices = sensitivity(case, case.nominal, base_samples=16, seed=20261031)
print("Filas base Sobol:", larger_indices["base_samples"])
print("Evaluaciones del motor Sobol:", larger_indices["evaluations"])
print("Estado Sobol:", larger_indices["status"])
if larger_indices["status"] != "computed":
    raise RuntimeError("El diseño de sensibilidad contiene sorteos rechazados")
effects = larger_indices["indices"]["recovery_pct"]
if effects.get("constant"):
    raise RuntimeError("La recuperación es constante con la tolerancia declarada")
display(pd.DataFrame({key: effects[key] for key in ("S1", "ST", "ST_conf")}))


Graficar índices totales de ambos presupuestos con bootstrap. Nombres y escala adimensional compartidos permiten comparar desplazamientos. No normalizar para sumar uno: los efectos totales incluyen interacciones y no deben necesariamente sumar unidad. El conteo exacto se imprime aparte para mostrar costo de estabilizar la estimación.

In [ ]:
# Comparar sensibilidad de orden total con ambos presupuestos, incluidas las semiamplitudes
# bootstrap de producción..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
names = list(larger_indices["inputs"])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for offset, record in ((-0.1, small_indices), (0.1, larger_indices)):
    effect = record["indices"]["recovery_pct"]
    values = [effect["ST"][name] for name in names]
    widths = [effect["ST_conf"][name] for name in names]
    ax.errorbar(
        values,
        np.arange(len(names)) + offset,
        xerr=widths,
        fmt="o",
        label="N=" + str(record["base_samples"]),
    )
ax.set_yticks(
    range(len(names)),
    [factor_labels.get(factor_name, factor_name) for factor_name in names],
    rotation=0,
)
ax.set(
    xlabel="Índice Sobol\ntotal (1)",
    ylabel="",
    title="Estabilidad de sensibilidad en diseño\nfinito",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


Intervalos amplios superpuestos impiden ordenar con confianza aun con estimaciones puntuales ordenadas. Un efecto grande consistente sugiere examinar el factor bajo estos supuestos, no establece un impulsor causal independiente. La lección integra distribución, rechazos y estabilidad finita. Decidir requiere diseños mayores repetidos y distribuciones conjuntas defendibles antes de citar riesgo u orden robustos.

Conviene distinguir cantidad de filas intentadas y recuperaciones resueltas. Si todas se resuelven, coinciden denominadores y es fácil olvidar la diferencia. Si un estado se rechaza, condicionar toda restricción a éxito haría parecer más confiable la operación de lo que sostiene el diseño. El propagador conserva rechazos y calcula factibilidad conjunta sobre intentos. No multiplicar probabilidades marginales para reconstruir la conjunta: ley, potencia y agua se acoplan mediante los mismos sorteos. La tabla de cuantiles describe por su parte salidas resueltas; no asigna recuperación cero a un estado para el cual el modelo no encontró solución física.

Un índice de sensibilidad grande tampoco implica gran dispersión absoluta. Puede explicar casi toda una varianza muy pequeña. Por eso se conserva primero la tabla de cuantiles y desviación de cada salida y después la descomposición adimensional. Al cambiar de recuperación a energía se debe volver a comprobar constancia numérica antes de interpretar un orden de factores.

## Resultados

**Respuesta:** El intervalo <!-- result:P05 de recuperación (%) -->0<!-- /result --> a <!-- result:P95 de recuperación (%) -->0<!-- /result --> por ciento usa <!-- result:Cantidad resuelta -->0<!-- /result --> recuperaciones resueltas; factibilidad <!-- result:Factibilidad conjunta (1) -->0<!-- /result --> incluye intentos y <!-- result:Cantidad rechazada -->0<!-- /result --> rechazos. El diseño Sobol mayor conserva semiamplitud máxima <!-- result:Máxima semiamplitud de efecto total (1) -->0<!-- /result -->; un orden puntual prolijo no certifica estabilidad de sensibilidad.

Resumir denominadores de propagación y semiamplitud del diseño Sobol mayor para interpretar presupuesto finito.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
answer_values = {
    "P05 de recuperación (%)": experiment["outputs"]["recovery_pct"]["p05"],
    "P95 de recuperación (%)": experiment["outputs"]["recovery_pct"]["p95"],
    "Cantidad resuelta": len(experiment["solved"]),
    "Cantidad rechazada": sum(experiment["refused"].values()),
    "Factibilidad conjunta (1)": experiment["probabilities"]["all_constraints"],
}
order_small = sorted(
    small_indices["inputs"], key=small_indices["indices"]["recovery_pct"]["ST"].get
)
order_large = sorted(
    larger_indices["inputs"], key=larger_indices["indices"]["recovery_pct"]["ST"].get
)
answer_values.update(
    {
        "Mismo orden total (1=sí)": int(order_small == order_large),
        "Máxima semiamplitud de efecto total (1)": max(
            larger_indices["indices"]["recovery_pct"]["ST_conf"].values()
        ),
    }
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

Los denominadores de cuantiles y factibilidad responden preguntas diferentes. Los presupuestos finitos agregan variabilidad e interacciones; sus intervalos acompañan estimaciones antes de interpretar un orden.

## Referencias

- https://doi.org/10.1080/00401706.1979.10489755